# Advanced Architecture

## 1. Multiple Inheritance and The Diamond Problem
Let's say we are building a smart home system. We have a generic Device, a SmartSpeaker, and a SmartClock. Finally, we want to build an AlarmClockEcho that acts as both.

In [6]:
class Device:
    def boot_up(self):
        return "Booting basic device..."

class SmartSpeaker(Device):
    def boot_up(self):
        return "Connecting to WiFi for speaker..."

class SmartClock(Device):
    def boot_up(self):
        return "Syncing time with NTP server..."

# Multiple Inheritance!
class AlarmClockEcho(SmartSpeaker, SmartClock):
    pass 

# The big question:
my_echo = AlarmClockEcho()
print(my_echo.boot_up())

Connecting to WiFi for speaker...


#### The Problem: 
AlarmClockEcho inherits boot_up() from SmartSpeaker, and it inherits boot_up() from SmartClock. They both inherited it from Device. This forms a diamond shape in the inheritance tree.

When you call my_echo.boot_up(), which one does Python execute?

## 2. Method Resolution Order (MRO)
Python solves this using a strict, predictable algorithm called MRO (Method Resolution Order). Specifically, modern Python uses the C3 Linearization algorithm.

You don't need to memorize the math behind the algorithm, but you must know its general rules:

Children precede parents: A subclass is always checked before its base classes.

Left-to-right: When a class inherits from multiple parents, they are checked in the order they are listed in the parentheses. (e.g., in class Child(ParentA, ParentB):, ParentA is checked before ParentB).

Let's see what the MRO is for our AlarmClockEcho. You can inspect this at any time using the special .__mro__ attribute or the .mro() method.


In [7]:
# Let's ask Python what the MRO is
print(AlarmClockEcho.__mro__)

(<class '__main__.AlarmClockEcho'>, <class '__main__.SmartSpeaker'>, <class '__main__.SmartClock'>, <class '__main__.Device'>, <class 'object'>)


So, when we call my_echo.boot_up(), Python checks:

1) Does AlarmClockEcho have the method? No.

2) Does SmartSpeaker have the method? Yes.

3) Python executes SmartSpeaker.boot_up() and stops looking.

## 3. Mixins: The Right Way to Use Multiple Inheritance
Because deep inheritance trees get incredibly complex and hard to debug, modern Python developers rarely use multiple inheritance to combine massive base classes.

Instead, they use a design pattern called Mixins.

A Mixin is a small class designed to add a very specific piece of functionality to another class. Mixins are not meant to stand on their own; you never instantiate a Mixin directly.

Imagine we are building a backend for an e-commerce site using Python (similar to what you might do with Spring Boot or FastAPI). We have different types of data models. We want some models to automatically convert to JSON, and others to log to the console when saved.

In [ ]:
# --- The Mixins ---
class JSONMixin:
    """Provides a specific utility to convert objects to JSON dictionaries."""
    def to_json(self):
        # We rely on the fact that any object this is mixed into 
        # will have a __dict__ attribute.
        import json
        return json.dumps(self.__dict__)

class AuditLoggingMixin:
    """Provides a utility to log saves."""
    def save(self):
        print(f"AUDIT LOG: Saving {self.__class__.__name__} to database...")
        super().save() # Pass the call up the MRO chain!

# --- The Base Class ---
class DatabaseModel:
    def save(self):
        print("Executing SQL INSERT...")

# --- The Concrete Class combining them ---
# We list Mixins FIRST so they take priority in the MRO!
class User(JSONMixin, AuditLoggingMixin, DatabaseModel):
    def __init__(self, username, email):
        self.username = username
        self.email = email

# Let's use it
new_user = User("shubhamj", "shubham@example.com")

# 1. We use the JSON functionality injected by the Mixin
print(new_user.to_json()) 
# Output: {"username": "subhamj", "email": "subham@example.com"}

# 2. When we call save(), the AuditLoggingMixin intercepts it, 
# logs the message, and then uses super() to pass it to the DatabaseModel!
new_user.save()
# Output: 
# AUDIT LOG: Saving User to database...
# Executing SQL INSERT...

{"username": "subhamj", "email": "subham@example.com"}
AUDIT LOG: Saving User to database...
Executing SQL INSERT...


### Why Mixins are beautiful: 
They keep your code modular. If you decide a Product model needs JSON support but not audit logging, you just inherit (JSONMixin, DatabaseModel). You compose behaviors exactly as you need them without messy parent-child relationships.